# Clef Employee Knowledge Assistant
## Phase 2 — Data Cleaning

This notebook preprocesses the selected Clef Employee Handbook documents before they are used for semantic chunking and RAG.

### Objectives

1. Load the selected Markdown documents.
2. Preserve the original documents.
3. Remove unnecessary Markdown/encoding artifacts.
4. Normalize whitespace and formatting.
5. Preserve headings and document structure.
6. Attach useful document metadata.
7. Generate cleaned documents.
8. Generate a cleaning report.

### Important principle

This is **loss-minimizing cleaning**.

We clean formatting and technical noise, but we do NOT rewrite,
summarize, interpret, or change the meaning of company policies.

Original files remain untouched.

## 1. Define the Project Directories

The project uses three important stages:

- `handbook/` → original GitHub repository (never modified)
- `processed/` → documents selected for our employee knowledge base
- `cleaned/` → cleaned versions produced by this notebook

Keeping these stages separate makes the preprocessing reproducible
and allows us to return to the original data whenever necessary.

In [63]:
!pip install pandas

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.8 MB 4.2 MB/s eta 0:00:03
   -- ------------------------------------- 0.5/9.8 MB 4.2 MB/s eta 0:00:03
   ---- ----------------------------------- 1.0/9.8 MB 1.5 MB/s eta 0:00:06
   ---- ----------------------------------- 1.0/9.8 MB 1.5 MB/s eta 0:00:06
   ----- ---------------------------------- 1.3/9.8 MB 1.2 MB/s eta 0:00:07
   ----- ---------------------------------- 1.3/9.8 MB 1.2 MB/s eta 0:00:07
   ----- ---------------------------------- 1.3/9.8 MB 1.2 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/9.8 MB 986.7 kB/s eta 0:00:09
   -------- ------------------------------- 2.1/9.8 MB 1.1 MB/s eta 0:00:08
   ---------- ----------------------------- 2.6/9.8 MB 1.2 MB/s eta 0:00:07
   ----------- ---------------------------- 2.9/9.8 MB 1.2 MB/s eta 0:00:06
   ------------- -------------------------- 3.4/9.8 MB 1.3 MB/s eta 0:00:06
   --------------

## 1. Import Modules

In [ ]:
from pathlib import Path
import re
import json
import html
import unicodedata
from collections import Counter

## 2. Determine Paths

In [ ]:
# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(r"E:\Projects\Speech AI\Data")

PROCESSED_DIR = PROJECT_ROOT / "processed"
CLEANED_DIR = PROCESSED_DIR / "cleaned"
PROCESSED_DIR = PROCESSED_DIR / "processed_raw"
METADATA_DIR = PROJECT_ROOT / "metadata"

# Create output directories if they don't exist
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print("Processed directory:", PROCESSED_DIR)
print("Cleaned directory:", CLEANED_DIR)
print("Metadata directory:", METADATA_DIR)

Processed directory: E:\Projects\Speech AI\Data\processed\processed_raw
Cleaned directory: E:\Projects\Speech AI\Data\processed\cleaned
Metadata directory: E:\Projects\Speech AI\Data\metadata


## 3. Inspect the Selected Documents

Before cleaning anything, we first inspect what is actually present
in the processed dataset.

This helps us catch accidental inclusion/exclusion of files before
running preprocessing.

In [44]:
# Find all Markdown files recursively
md_files = sorted(PROCESSED_DIR.rglob("*.md"))

print(f"Total Markdown files found: {len(md_files)}")
print()

for i, file_path in enumerate(md_files, start=1):
    relative_path = file_path.relative_to(PROCESSED_DIR)
    print(f"{i:02d}. {relative_path}")

Total Markdown files found: 31

01. Benefits and Perks\Continuing Education.md
02. Benefits and Perks\Healthcare and Disability Insurance.md
03. Benefits and Perks\Holiday List.md
04. Benefits and Perks\New Parent Leave.md
05. Benefits and Perks\Other Protected Absences.md
06. Benefits and Perks\Referral Bonuses.md
07. Benefits and Perks\Sabbatical.md
08. Benefits and Perks\Vacation and Sick Leave.md
09. Clef Values.md
10. Employment Policies\At-Will Employment.md
11. Employment Policies\Code of Conduct in the Community.md
12. Employment Policies\Complaint Policy.md
13. Employment Policies\Drug and Alcohol Policy.md
14. Employment Policies\Employee Privacy.md
15. Employment Policies\Equal Opportunity Employment.md
16. Employment Policies\Salary and Equity Compensation.md
17. Employment Policies\Working Remotely.md
18. Hiring Documents\Handbook Introduction.md
19. Mission Statement.md
20. Onboarding Documents\Communication and Transparency.md
21. Onboarding Documents\Direct Reports.md
2

## 4. Check the Dataset Structure
This is useful later when you want to analyze retrieval performance by category.

In [45]:
# Count documents by top-level category

category_counts = Counter()

for file_path in md_files:
    relative = file_path.relative_to(PROCESSED_DIR)
    
    if len(relative.parts) > 1:
        category = relative.parts[0]
    else:
        category = "Root"
    
    category_counts[category] += 1

print("Documents by category:\n")

for category, count in category_counts.items():
    print(f"{category}: {count}")

Documents by category:

Benefits and Perks: 8
Root: 4
Employment Policies: 8
Hiring Documents: 1
Onboarding Documents: 6
Operations Documents: 4


## 5. Inspect a Document Before Cleaning
Inspect Raw Markdown

Before writing the cleaning logic, let's look at an actual document.

This is important because cleaning should be based on the real dataset,
not assumptions about how the Markdown is formatted.

In [69]:
# Display the first document

sample_file = md_files[0]  # change the index to view different files

print("File:", sample_file.relative_to(PROCESSED_DIR))
print("=" * 80)

raw_text = sample_file.read_text(encoding="utf-8")

print(raw_text[:5000])

File: Benefits and Perks\Continuing Education.md
# Continuing Education

One of Clef’s core values is “Be better today than yesterday,” so it’s important that we support our employees’ efforts to learn, grow, and improve. These are some of the key benefits of working at Clef, and are central to our company culture.

## Learning Budget

Every employee has a company budget to support any learning activity that they want to pursue related to the work they do at Clef. This doesn’t need to be a class explicitly linked to their current role, but it should help them improve a skill that will be useful for them at Clef. Each employee has an annual budget of $4,000 which can be spent towards program fees/tuition, tickets, flights, and hotels for industry conferences, classes, mentorship programs, books, programs, videos, or other places that they feel will provide valuable learning experiences. These expenses should be discussed in one-on-ones and approved by the founder to whom the employee re

## 6. Cleaning Strategy

We will perform only safe transformations.

### We WILL:

- Normalize Unicode characters.
- Normalize line endings.
- Remove unnecessary escape characters.
- Decode HTML entities.
- Normalize excessive whitespace.
- Clean Markdown links while preserving their visible text.
- Preserve headings.
- Preserve paragraphs and lists.
- Preserve the actual policy wording.

### We WILL NOT:

- Summarize documents.
- Rewrite policies.
- Correct policy meaning.
- Delete potentially useful sections.
- Automatically remove tables/forms.
- Automatically remove navigation sections.

Those decisions will be handled later during structural preprocessing
and chunking.

In [47]:
def normalize_unicode(text):
    """
    Normalize Unicode characters without converting
    the document to ASCII.
    """
    return unicodedata.normalize("NFKC", text)


def normalize_line_endings(text):
    """
    Convert Windows/macOS line endings into Unix-style line endings.
    """
    return text.replace("\r\n", "\n").replace("\r", "\n")


def decode_html_entities(text):
    """
    Convert HTML entities such as &amp; into their
    corresponding characters.
    """
    return html.unescape(text)


def clean_markdown_links(text):
    """
    Convert Markdown links:

        [Employee Privacy](url)

    into:

        Employee Privacy

    The visible link text is preserved.
    """
    pattern = r"\[([^\]]+)\]\([^)]+\)"
    return re.sub(pattern, r"\1", text)


def clean_unnecessary_escapes(text):
    """
    Remove Markdown escapes where they are unnecessary
    for our retrieval representation.

    Example:
        \# Heading -> # Heading
        \& -> &
    """
    text = re.sub(r"\\([#*_`~&:;.!?()\[\]{}])", r"\1", text)
    return text


def normalize_whitespace(text):
    """
    Remove excessive spaces while preserving paragraph structure.
    """

    # Remove trailing spaces
    text = re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)

    # Convert 3+ blank lines into a maximum of 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

<>:46: SyntaxWarning: invalid escape sequence '\#'
<>:46: SyntaxWarning: invalid escape sequence '\#'
C:\Users\DELL\AppData\Local\Temp\ipykernel_29548\4017584486.py:46: SyntaxWarning: invalid escape sequence '\#'
  \# Heading -> # Heading


## 7. Main Cleaning Function

In [48]:
def clean_document(text):
    """
    Apply all safe cleaning operations in a controlled order.
    """

    cleaned = text

    # 1. Unicode normalization
    cleaned = normalize_unicode(cleaned)

    # 2. Line ending normalization
    cleaned = normalize_line_endings(cleaned)

    # 3. Decode HTML entities
    cleaned = decode_html_entities(cleaned)

    # 4. Clean Markdown links
    cleaned = clean_markdown_links(cleaned)

    # 5. Remove unnecessary Markdown escapes
    cleaned = clean_unnecessary_escapes(cleaned)

    # 6. Normalize whitespace
    cleaned = normalize_whitespace(cleaned)

    return cleaned

## 8. Test Cleaning on One Document

Test the Cleaning Function

Before processing all documents, test the cleaning function on one file.

This is a safety step. We want to visually compare the original
and cleaned versions before applying the transformation to the
whole dataset.

In [58]:
cleaned_sample = clean_document(raw_text)

print("ORIGINAL")
print("=" * 80)
print(raw_text[:5000])

print("\n\nCLEANED")
print("=" * 80)
print(cleaned_sample[:5000])

ORIGINAL
# Continuing Education

One of Clef’s core values is “Be better today than yesterday,” so it’s important that we support our employees’ efforts to learn, grow, and improve. These are some of the key benefits of working at Clef, and are central to our company culture.

## Learning Budget

Every employee has a company budget to support any learning activity that they want to pursue related to the work they do at Clef. This doesn’t need to be a class explicitly linked to their current role, but it should help them improve a skill that will be useful for them at Clef. Each employee has an annual budget of $4,000 which can be spent towards program fees/tuition, tickets, flights, and hotels for industry conferences, classes, mentorship programs, books, programs, videos, or other places that they feel will provide valuable learning experiences. These expenses should be discussed in one-on-ones and approved by the founder to whom the employee reports. This budget resets at the beginni

## 9. Add Document Metadata

Generate Document Metadata

Every document will receive metadata.

This metadata will later be inherited by its chunks and stored
alongside the vector embeddings.

Example:

    document_id
    title
    category
    source_file
    access_level

The access level represents our project rule:

> Only documents available to all employees belong in the
> employee knowledge base.

In [59]:
def create_document_id(file_path):
    """
    Create a stable document ID from the file path.
    """
    relative = file_path.relative_to(PROCESSED_DIR)

    # Remove extension and normalize
    name = relative.with_suffix("").as_posix()

    name = name.lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = name.strip("_")

    return name


def get_document_title(file_path, text):
    """
    Prefer the first Markdown H1 as the document title.
    Otherwise use the filename.
    """

    # Look for first H1
    match = re.search(r"^\s*#\s+(.+?)\s*$", text, re.MULTILINE)

    if match:
        return match.group(1).strip()

    # Fallback to filename
    return file_path.stem


def get_category(file_path):
    """
    Extract the top-level category from the folder structure.
    """

    relative = file_path.relative_to(PROCESSED_DIR)

    if len(relative.parts) > 1:
        return relative.parts[0]

    return "General"

## 10. Process All Documents

In [60]:
documents_metadata = []
processing_stats = []

for file_path in md_files:

    # Read original
    original_text = file_path.read_text(encoding="utf-8")

    # Clean
    cleaned_text = clean_document(original_text)

    # Metadata
    document_id = create_document_id(file_path)
    title = get_document_title(file_path, cleaned_text)
    category = get_category(file_path)

    relative_path = file_path.relative_to(PROCESSED_DIR)

    metadata = {
        "document_id": document_id,
        "title": title,
        "category": category,
        "access_level": "employee",
        "source_file": str(relative_path).replace("\\", "/")
    }

    documents_metadata.append(metadata)

    # Preserve folder structure in cleaned directory
    output_path = CLEANED_DIR / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)

    output_path.write_text(
        cleaned_text,
        encoding="utf-8"
    )

    # Statistics
    processing_stats.append({
        "document_id": document_id,
        "source_file": str(relative_path).replace("\\", "/"),
        "original_characters": len(original_text),
        "cleaned_characters": len(cleaned_text),
        "characters_removed": len(original_text) - len(cleaned_text)
    })

print(f"Processed {len(documents_metadata)} documents.")

Processed 31 documents.


## 11. Save Metadata

In [61]:
metadata_path = METADATA_DIR / "documents.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(
        documents_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Metadata saved to: {metadata_path}")

Metadata saved to: E:\Projects\Speech AI\Data\metadata\documents.json


## 12. Generate Cleaning Report
Cleaning Report

We now compare the original and cleaned documents.

A very large reduction in text length could indicate that our
cleaning logic accidentally removed meaningful content, so this
report acts as a basic quality-control step.

In [70]:
import pandas as pd

stats_df = pd.DataFrame(processing_stats)

stats_df["reduction_percent"] = (
    stats_df["characters_removed"]
    / stats_df["original_characters"]
    * 100
)

stats_df = stats_df.sort_values(
    "reduction_percent",
    ascending=False
)

stats_df

,document_id,source_file,original_characters,cleaned_characters,characters_removed,reduction_percent
30,readme,README.md,7052,3104,3948,55.984118
7,benefits_and_perks_vacation_and_sick_leave,Benefits and Perks/Vacation and Sick Leave.md,984,894,90,9.146341
28,operations_documents_sharing_files,Operations Documents/Sharing Files.md,1428,1351,77,5.392157
29,policy_changes,Policy Changes.md,6371,6123,248,3.892639
15,employment_policies_salary_and_equity_compensa...,Employment Policies/Salary and Equity Compensa...,3084,2985,99,3.210117
16,employment_policies_working_remotely,Employment Policies/Working Remotely.md,6860,6645,215,3.134111
10,employment_policies_code_of_conduct_in_the_com...,Employment Policies/Code of Conduct in the Com...,2153,2119,34,1.579192
24,onboarding_documents_welcome_to_clef,Onboarding Documents/Welcome to Clef.md,5873,5805,68,1.157841
17,hiring_documents_handbook_introduction,Hiring Documents/Handbook Introduction.md,897,888,9,1.003344
4,benefits_and_perks_other_protected_absences,Benefits and Perks/Other Protected Absences.md,1733,1728,5,0.288517


In [75]:
print(stats_df["original_characters"].sum())
print(stats_df["cleaned_characters"].sum())

79415
74583


## 13. Look for Suspicious Cleaning


In [76]:
# Documents where more than 10% of characters were removed

suspicious = stats_df[
    stats_df["reduction_percent"] > 10
]

print(
    f"Documents with >10% character reduction: "
    f"{len(suspicious)}"
)

suspicious

Documents with >10% character reduction: 1


,document_id,source_file,original_characters,cleaned_characters,characters_removed,reduction_percent
30,readme,README.md,7052,3104,3948,55.984118


## 14. Verify Document Count

In [77]:
cleaned_files = sorted(CLEANED_DIR.rglob("*.md"))

print("Input documents :", len(md_files))
print("Output documents:", len(cleaned_files))

assert len(md_files) == len(cleaned_files), \
    "Document count mismatch!"

print("✓ Document count verified.")

Input documents : 31
Output documents: 31
✓ Document count verified.


## 15. Verify No Empty Documents

In [78]:
empty_documents = []

for file_path in cleaned_files:

    text = file_path.read_text(encoding="utf-8").strip()

    if not text:
        empty_documents.append(
            file_path.relative_to(CLEANED_DIR)
        )

if empty_documents:
    print("WARNING: Empty documents found:")
    for doc in empty_documents:
        print("-", doc)
else:
    print("✓ No empty documents found.")

✓ No empty documents found.


## 16. Verify Headings Were Preserved

In [79]:
heading_pattern = re.compile(
    r"^\s*#{1,6}\s+.+$",
    re.MULTILINE
)

heading_report = []

for file_path in cleaned_files:

    text = file_path.read_text(encoding="utf-8")

    headings = heading_pattern.findall(text)

    heading_report.append({
        "file": str(
            file_path.relative_to(CLEANED_DIR)
        ).replace("\\", "/"),
        "heading_count": len(headings)
    })

heading_df = pd.DataFrame(heading_report)

heading_df.sort_values(
    "heading_count",
    ascending=False
).head(20)

,file,heading_count
16,Employment Policies/Working Remotely.md,13
19,Onboarding Documents/Communication and Transpa...,13
29,Policy Changes.md,12
13,Employment Policies/Employee Privacy.md,9
30,README.md,8
24,Onboarding Documents/Welcome to Clef.md,6
26,Operations Documents/Effective Meetings.md,6
8,Clef Values.md,5
4,Benefits and Perks/Other Protected Absences.md,5
22,Onboarding Documents/One on Ones.md,4


## 17. Final Inspection

## 8. Final Manual Inspection

Before moving to chunking, manually inspect several cleaned documents.

Recommended:

1. One Benefits document
2. One Employment Policy
3. One Onboarding document
4. One Operations document
5. One document containing lists/links/forms

The cleaned document should:

- Preserve the original policy meaning.
- Preserve headings.
- Preserve lists.
- Preserve important numbers/dates.
- Preserve paragraphs.
- Remove only formatting/encoding noise.

If the cleaned version looks correct, the dataset is ready for
semantic/structure-aware chunking.

In [80]:
# Display a chosen cleaned document

inspect_file = cleaned_files[0]

print("FILE:")
print(inspect_file.relative_to(CLEANED_DIR))
print("\n" + "=" * 80)

print(
    inspect_file.read_text(encoding="utf-8")
)

FILE:
Benefits and Perks\Continuing Education.md

# Continuing Education

One of Clef’s core values is “Be better today than yesterday,” so it’s important that we support our employees’ efforts to learn, grow, and improve. These are some of the key benefits of working at Clef, and are central to our company culture.

## Learning Budget

Every employee has a company budget to support any learning activity that they want to pursue related to the work they do at Clef. This doesn’t need to be a class explicitly linked to their current role, but it should help them improve a skill that will be useful for them at Clef. Each employee has an annual budget of $4,000 which can be spent towards program fees/tuition, tickets, flights, and hotels for industry conferences, classes, mentorship programs, books, programs, videos, or other places that they feel will provide valuable learning experiences. These expenses should be discussed in one-on-ones and approved by the founder to whom the employee r

In [82]:
report_path = METADATA_DIR / "cleaning_report.csv"

stats_df.to_csv(
    report_path,
    index=False,
    encoding="utf-8"
)

print(f"Cleaning report saved to: {report_path}")

Cleaning report saved to: E:\Projects\Speech AI\Data\metadata\cleaning_report.csv


### Understanding/verifing functions

In [33]:
text = (
    "  # Employee Privacy Policy  \r\n"
    "\r\n"
    "Our company values privacy &amp; security.   \r\n"
    "Employees should review the [Employee Privacy](https://example.com/privacy) policy.  \r\n"
    "\r\n"
    "\r\n"
    "The policy applies to employees in India ₹ and Europe é.   \r\n"
    "\\# Important: Follow the rules \\& regulations.  \r\n"
    "\\* Do not share confidential information.①   \r\n"
    "\r\n"
    "\r\n"
    "\r\n"
    "Next section →  "
)

In [34]:
print("=" * 70)
print("ORIGINAL TEXT")
print("=" * 70)
print(repr(text))
print(text)

ORIGINAL TEXT
'  # Employee Privacy Policy  \r\n\r\nOur company values privacy &amp; security.   \r\nEmployees should review the [Employee Privacy](https://example.com/privacy) policy.  \r\n\r\n\r\nThe policy applies to employees in India ₹ and Europe é.   \r\n\\# Important: Follow the rules \\& regulations.  \r\n\\* Do not share confidential information.①   \r\n\r\n\r\n\r\nNext section →  '
  # Employee Privacy Policy  

Our company values privacy &amp; security.   
Employees should review the [Employee Privacy](https://example.com/privacy) policy.  


The policy applies to employees in India ₹ and Europe é.   
\# Important: Follow the rules \& regulations.  
\* Do not share confidential information.①   



Next section →  


In [35]:
text = normalize_unicode(text)

print("\n" + "=" * 70)
print("AFTER normalize_unicode()")
print("=" * 70)
print(repr(text))
print(text)



AFTER normalize_unicode()
'  # Employee Privacy Policy  \r\n\r\nOur company values privacy &amp; security.   \r\nEmployees should review the [Employee Privacy](https://example.com/privacy) policy.  \r\n\r\n\r\nThe policy applies to employees in India ₹ and Europe é.   \r\n\\# Important: Follow the rules \\& regulations.  \r\n\\* Do not share confidential information.1   \r\n\r\n\r\n\r\nNext section →  '
  # Employee Privacy Policy  

Our company values privacy &amp; security.   
Employees should review the [Employee Privacy](https://example.com/privacy) policy.  


The policy applies to employees in India ₹ and Europe é.   
\# Important: Follow the rules \& regulations.  
\* Do not share confidential information.1   



Next section →  


In [36]:
# ============================================================
# 2. Line Ending Normalization
# ============================================================

text = normalize_line_endings(text)

print("\n" + "=" * 70)
print("AFTER normalize_line_endings()")
print("=" * 70)
print(repr(text))
print(text)


AFTER normalize_line_endings()
'  # Employee Privacy Policy  \n\nOur company values privacy &amp; security.   \nEmployees should review the [Employee Privacy](https://example.com/privacy) policy.  \n\n\nThe policy applies to employees in India ₹ and Europe é.   \n\\# Important: Follow the rules \\& regulations.  \n\\* Do not share confidential information.1   \n\n\n\nNext section →  '
  # Employee Privacy Policy  

Our company values privacy &amp; security.   
Employees should review the [Employee Privacy](https://example.com/privacy) policy.  


The policy applies to employees in India ₹ and Europe é.   
\# Important: Follow the rules \& regulations.  
\* Do not share confidential information.1   



Next section →  


In [37]:
# ============================================================
# 3. HTML Entity Decoding
# ============================================================

text = decode_html_entities(text)

print("\n" + "=" * 70)
print("AFTER decode_html_entities()")
print("=" * 70)
print(repr(text))
print(text)


AFTER decode_html_entities()
'  # Employee Privacy Policy  \n\nOur company values privacy & security.   \nEmployees should review the [Employee Privacy](https://example.com/privacy) policy.  \n\n\nThe policy applies to employees in India ₹ and Europe é.   \n\\# Important: Follow the rules \\& regulations.  \n\\* Do not share confidential information.1   \n\n\n\nNext section →  '
  # Employee Privacy Policy  

Our company values privacy & security.   
Employees should review the [Employee Privacy](https://example.com/privacy) policy.  


The policy applies to employees in India ₹ and Europe é.   
\# Important: Follow the rules \& regulations.  
\* Do not share confidential information.1   



Next section →  


In [38]:
# ============================================================
# 4. Markdown Link Cleaning
# ============================================================

text = clean_markdown_links(text)

print("\n" + "=" * 70)
print("AFTER clean_markdown_links()")
print("=" * 70)
print(repr(text))
print(text)


AFTER clean_markdown_links()
'  # Employee Privacy Policy  \n\nOur company values privacy & security.   \nEmployees should review the Employee Privacy policy.  \n\n\nThe policy applies to employees in India ₹ and Europe é.   \n\\# Important: Follow the rules \\& regulations.  \n\\* Do not share confidential information.1   \n\n\n\nNext section →  '
  # Employee Privacy Policy  

Our company values privacy & security.   
Employees should review the Employee Privacy policy.  


The policy applies to employees in India ₹ and Europe é.   
\# Important: Follow the rules \& regulations.  
\* Do not share confidential information.1   



Next section →  


In [39]:
# ============================================================
# 5. Unnecessary Markdown Escape Cleaning
# ============================================================

text = clean_unnecessary_escapes(text)

print("\n" + "=" * 70)
print("AFTER clean_unnecessary_escapes()")
print("=" * 70)
print(repr(text))
print(text)


AFTER clean_unnecessary_escapes()
'  # Employee Privacy Policy  \n\nOur company values privacy & security.   \nEmployees should review the Employee Privacy policy.  \n\n\nThe policy applies to employees in India ₹ and Europe é.   \n# Important: Follow the rules & regulations.  \n* Do not share confidential information.1   \n\n\n\nNext section →  '
  # Employee Privacy Policy  

Our company values privacy & security.   
Employees should review the Employee Privacy policy.  


The policy applies to employees in India ₹ and Europe é.   
# Important: Follow the rules & regulations.  
* Do not share confidential information.1   



Next section →  


In [40]:
# ============================================================
# 6. Whitespace Normalization
# ============================================================

text = normalize_whitespace(text)

print("\n" + "=" * 70)
print("AFTER normalize_whitespace()")
print("=" * 70)
print(repr(text))
print(text)


AFTER normalize_whitespace()
'# Employee Privacy Policy\n\nOur company values privacy & security.\nEmployees should review the Employee Privacy policy.\n\nThe policy applies to employees in India ₹ and Europe é.\n# Important: Follow the rules & regulations.\n* Do not share confidential information.1\n\nNext section →'
# Employee Privacy Policy

Our company values privacy & security.
Employees should review the Employee Privacy policy.

The policy applies to employees in India ₹ and Europe é.
# Important: Follow the rules & regulations.
* Do not share confidential information.1

Next section →


Verification complete